# 05 — Hyperparameter Optimization & Sensitivity Analysis

**Tuning Tree Depth, Regularization, and Learning Rates**

This notebook investigates how key hyperparameters affect model performance and illustrates overfitting, underfitting, and the optimal operating point for three model families: Decision Tree, Random Forest, and LightGBM.

In [ ]:
import sys
sys.path.insert(0, '.')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import warnings
warnings.filterwarnings("ignore")

from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.metrics import f1_score, fbeta_score

try:
    import lightgbm as lgb
    HAS_LGBM = True
except ImportError:
    HAS_LGBM = False
    print("[WARNING] lightgbm not installed; LightGBM sections will be skipped.")

sns.set_theme(style='whitegrid')
%matplotlib inline

from src.features import FEATURE_COLUMNS
feat_df = pd.read_csv('data/processed/feature_matrix.csv')
X = feat_df[FEATURE_COLUMNS].values
y = feat_df['label'].values

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

print(f'Train: {X_train.shape}, Val: {X_val.shape}')
print(f'Positive rate — train: {y_train.mean():.3f}, val: {y_val.mean():.3f}')

### 1. Decision Tree Depth Sensitivity — Overfitting Analysis

A Decision Tree with unlimited depth memorizes the training set (overfitting). By sweeping `max_depth` we find where validation accuracy peaks and the gap between train and validation accuracy begins to widen (the **overfitting zone**).

In [ ]:
depths = list(range(2, 21, 2))
train_accs, val_accs = [], []

for d in depths:
    clf = DecisionTreeClassifier(max_depth=d, random_state=42)
    clf.fit(X_train, y_train)
    train_accs.append(clf.score(X_train, y_train))
    val_accs.append(clf.score(X_val, y_val))

best_depth = depths[int(np.argmax(val_accs))]

plt.figure(figsize=(9, 5))
plt.plot(depths, train_accs, 'o-', color='#2980b9', linewidth=2, label='Train Accuracy')
plt.plot(depths, val_accs,   's--', color='#e74c3c', linewidth=2, label='Val Accuracy')
plt.axvline(best_depth, color='gray', linestyle=':', linewidth=1.5,
            label=f'Best depth = {best_depth}')
plt.xlabel('max_depth', fontsize=12)
plt.ylabel('Accuracy', fontsize=12)
plt.title('Decision Tree: Depth vs Accuracy (Overfitting Curve)', fontsize=13, fontweight='bold')
plt.legend()
plt.xticks(depths)
plt.tight_layout()
os.makedirs('results/plots', exist_ok=True)
plt.savefig('results/plots/dt_depth_sensitivity.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Best validation accuracy at max_depth={best_depth}: {max(val_accs):.4f}')

### 2. Random Forest — n_estimators Sensitivity

More trees reduce variance (via averaging) but have diminishing returns. We plot OOF F0.5 as a function of `n_estimators` to find the efficiency plateau.

In [ ]:
n_estimators_list = [10, 25, 50, 100, 200]
f05_scores = []

skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)

for n_est in n_estimators_list:
    rf = RandomForestClassifier(n_estimators=n_est, n_jobs=-1, random_state=42)
    scores = cross_val_score(rf, X_train, y_train, cv=skf,
                             scoring='f1', n_jobs=-1)
    f05_scores.append(scores.mean())
    print(f'  n_estimators={n_est:>4d}  OOF-F1={scores.mean():.4f} ± {scores.std():.4f}')

plt.figure(figsize=(9, 5))
plt.plot(n_estimators_list, f05_scores, 'o-', color='#27ae60',
         linewidth=2.5, markersize=8)
plt.xlabel('n_estimators', fontsize=12)
plt.ylabel('OOF F1 Score (3-Fold)', fontsize=12)
plt.title('Random Forest: n_estimators vs OOF F1', fontsize=13, fontweight='bold')
plt.xticks(n_estimators_list)
plt.tight_layout()
plt.savefig('results/plots/rf_n_estimators_sensitivity.png', dpi=150, bbox_inches='tight')
plt.show()

### 3. LightGBM — Learning Rate vs num_leaves Grid

`learning_rate` controls the step size in gradient boosting; `num_leaves` controls model complexity. The optimal combination balances expressiveness with regularization. We sweep a 3×3 grid and show F0.5 as a heat map.

In [ ]:
if not HAS_LGBM:
    print('[SKIP] LightGBM not installed.')
else:
    learning_rates = [0.01, 0.05, 0.1]
    num_leaves_list = [15, 31, 63]

    grid_f05 = np.zeros((len(learning_rates), len(num_leaves_list)))

    for i, lr in enumerate(learning_rates):
        for j, nl in enumerate(num_leaves_list):
            clf = lgb.LGBMClassifier(
                n_estimators=50, learning_rate=lr, num_leaves=nl,
                random_state=42, n_jobs=-1, verbose=-1)
            clf.fit(X_train, y_train)
            y_pred = clf.predict(X_val)
            score = fbeta_score(y_val, y_pred, beta=0.5, average='macro', zero_division=0)
            grid_f05[i, j] = score
            print(f'  lr={lr}, num_leaves={nl} -> F0.5={score:.4f}')

    fig, ax = plt.subplots(figsize=(8, 5))
    sns.heatmap(grid_f05, annot=True, fmt='.4f', cmap='YlGn',
                xticklabels=num_leaves_list, yticklabels=learning_rates,
                ax=ax, linewidths=0.5, annot_kws={'size': 12})
    ax.set_xlabel('num_leaves', fontsize=12)
    ax.set_ylabel('learning_rate', fontsize=12)
    ax.set_title('LightGBM Grid Search: Val Macro F0.5', fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.savefig('results/plots/lgbm_grid_search.png', dpi=150, bbox_inches='tight')
    plt.show()

    best_idx = np.unravel_index(np.argmax(grid_f05), grid_f05.shape)
    best_lr = learning_rates[best_idx[0]]
    best_nl = num_leaves_list[best_idx[1]]
    print(f'\nBest: learning_rate={best_lr}, num_leaves={best_nl}, F0.5={grid_f05[best_idx]:.4f}')

### 4. Best Hyperparameters Summary

Consolidated table of the best hyperparameters found for each model family and the corresponding validation performance.

In [ ]:
best_params = [
    {
        'Model': 'Decision Tree',
        'Key Hyperparameter': 'max_depth',
        'Best Value': str(best_depth),
        'Val Metric': f'{max(val_accs):.4f} (Accuracy)',
    },
    {
        'Model': 'Random Forest',
        'Key Hyperparameter': 'n_estimators',
        'Best Value': str(n_estimators_list[int(np.argmax(f05_scores))]),
        'Val Metric': f'{max(f05_scores):.4f} (OOF F1)',
    },
]

if HAS_LGBM:
    best_params.append({
        'Model': 'LightGBM',
        'Key Hyperparameter': 'learning_rate / num_leaves',
        'Best Value': f'{best_lr} / {best_nl}',
        'Val Metric': f'{grid_f05[best_idx]:.4f} (Macro F0.5)',
    })

summary_df = pd.DataFrame(best_params)
summary_df.style\
    .set_properties(**{'text-align': 'left'})\
    .hide(axis='index')